# Stopping tables

Interpolate the beam-stopping cross section from tables instead of evaluating a model: tabulate a slow model once, save it, and reuse it; or use the effective stopping coefficients of ADAS (adf21 files).

`TabulatedStopping` holds one table $\sigma_i(E, n, T)$ per receiver species $i$, the plasma ion the beam collides with, on a grid of beam energy per nucleon, density and temperature. Between the nodes $\ln\sigma$ is linear in $\ln E$, $\ln n$ and $\ln T$. A plasma of several species is the composite of ADAS (Anderson et al., Plasma Phys. Control. Fusion 42, 781, 2000): each species stops the beam as in a pure plasma of it, weighted with its share of the electrons,

$$\sigma_s = \sum_i \frac{Z_i n_i}{\sum_j Z_j n_j}\,\sigma_i(E, n_i^\star, T).$$

Suzuki's fit has this form with every species read at $n_i^\star = n_e$ (`density_rule="electron"`); ADAS reads each species at the equivalent electron density $n_i^\star = \sum_j Z_j^2 n_j / Z_i$ (`density_rule="equivalent"`).

In [ ]:
import tempfile
import time
from pathlib import Path

import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge.stopping_tables import (
    Adf21,
    TabulatedStopping,
    read_adf21,
    write_adf21,
)

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX


def log_grid(start, stop, per_decade, extra=()):
    """Nodes equally spaced in the logarithm, plus ``extra``."""
    count = int(np.ceil(per_decade * np.log10(stop / start))) + 1
    return np.union1d(np.geomspace(start, stop, count), extra)


suzuki = foc.SuzukiStopping()
# 15 nodes per decade; nodes at the edges of Suzuki's blend window (kinks)
energies = log_grid(10e3, 5e6, 15, (1e5 / 1.1, 1.1e5))  # eV/amu
densities = log_grid(1e18, 3e20, 15)  # m^-3
temperatures = log_grid(100.0, 3e4, 15)  # eV
species = [(2, 1), (3, 1), (4, 2), (12, 6)]  # (mass number, charge): D, T, He, C

start = time.perf_counter()
table = TabulatedStopping.from_model(suzuki, energies, densities, temperatures, species)
print(f"tabulated in {time.perf_counter() - start:.2f} s")

folder = Path(tempfile.mkdtemp())
table.save(folder / "suzuki.nc")
table = TabulatedStopping.load(folder / "suzuki.nc")
table.table

`from_model` samples a hydrogen species in a pure plasma of it, and an impurity in a mixture with the first hydrogen species, from which its pure-plasma cross section follows by the composite rule. For Suzuki's fit that is exact, so the table reproduces the fit at its nodes. The table is a netCDF file: `TabulatedStopping.load` reads it back.

## Accuracy

Between the nodes the interpolation error depends on the curvature of $\ln\sigma$. The map is the relative error in a deuterium plasma with 2 % of the electrons from carbon, at $T_e$ = 5 keV, on a grid much finer than the table's:

In [ ]:
ions = {"D": (2, 1, 0.98), "C": (12, 6, 0.02 / 6)}  # mass number, charge, n_i / n_e


def ion_arguments(ne, ions):
    return (
        [share * ne for _, _, share in ions.values()],
        [a for a, _, _ in ions.values()],
        [z for _, z, _ in ions.values()],
    )


e_fine = np.geomspace(energies[0], energies[-1], 300)
n_fine = np.geomspace(densities[0], densities[-1], 200)
e, n = np.meshgrid(e_fine, n_fine, indexing="ij")
args = (e, n, 5e3, *ion_arguments(n, ions))
error = 100 * (np.asarray(table(*args)) / np.asarray(suzuki(*args)) - 1)

# and at random points of the whole grid, for several plasmas
rng = np.random.default_rng(1)


def log_uniform(nodes, size=200_000):
    return np.exp(rng.uniform(np.log(nodes[0]), np.log(nodes[-1]), size))


e3, n3, t3 = log_uniform(energies), log_uniform(densities), log_uniform(temperatures)
plasmas = {
    "D": {"D": (2, 1, 1.0)},
    "DT": {"D": (2, 1, 0.5), "T": (3, 1, 0.5)},
    "D + He": {"D": (2, 1, 0.9), "He": (4, 2, 0.05)},
    "D + C": ions,
}
for name, mix in plasmas.items():
    args3 = (e3, n3, t3, *ion_arguments(n3, mix))
    rel = np.abs(np.asarray(table(*args3)) / np.asarray(suzuki(*args3)) - 1)
    print(f"{name:7s} max {100 * rel.max():.2f} %, mean {100 * rel.mean():.3f} %")
    assert rel.max() < 5e-3

error_map = xr.DataArray(
    error,
    dims=("energy", "density"),
    coords={
        "energy": (
            "energy",
            np.log10(e_fine / 1e3),
            {"long_name": r"$\log_{10} E$ [keV/amu]"},
        ),
        "density": (
            "density",
            np.log10(n_fine),
            {"long_name": r"$\log_{10} n_e$ [m$^{-3}$]"},
        ),
    },
    name="error",
    attrs={"long_name": "table / Suzuki - 1", "units": "%"},
)
error_map.plasma.plot.slice(
    x="energy", y="density", symmetric=True, cmap="RdBu_r", title=""
)

The error vanishes at the nodes and stays below 0.5 % everywhere. It is largest near 100 keV/amu, inside the window where `SuzukiStopping` blends its two published fits and $\ln\sigma$ bends most; a finer energy grid there lowers it.

## Speed

The table needs a cell lookup and a trilinear interpolation per species and point, the fit a few dozen array operations and, with impurities, a selection of its coefficient rows:

In [ ]:
points = 1_000_000
e3, n3, t3 = (
    log_uniform(energies, points),
    log_uniform(densities, points),
    log_uniform(temperatures, points),
)


def best_time(model, args, repeat=3):
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        model(*args)
        times.append(time.perf_counter() - start)
    return min(times)


speedup = {}
for name in ("D", "D + C"):
    args3 = (e3, n3, t3, *ion_arguments(n3, plasmas[name]))
    t_model, t_table = best_time(suzuki, args3), best_time(table, args3)
    speedup[name] = t_model / t_table
    print(
        f"{name:6s}: Suzuki {1e3 * t_model:6.1f} ms, table {1e3 * t_table:6.1f} ms "
        f"for {points:.0e} points: {speedup[name]:.1f} times faster"
    )
assert speedup["D + C"] > 1.0

For a model that is itself expensive, such as a collisional-radiative calculation (`ElectronCRStopping` with OpenCR), the gain is the whole cost of the model: `from_model` runs it once on the grid, and the saved table is reused.

## In the deposition

The deposition only calls the stopping model, so the table plugs in for it. A 500 keV deuterium beam in an ITER-like Solov'ev equilibrium with parabolic profiles:

In [ ]:
equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
origin, direction = foc.tangential_injection(5.3, 12.0)
beam = foc.NeutralBeam.gaussian(
    origin, direction, 0.5e6, 16.5e6, width=0.3, divergence=0.005, markers=400, seed=1
)
plasma = foc.Plasma.parabolic(3e19, 1e4, ions=ions)
rho_edges = np.linspace(0.0, 1.0, 21)
results = {}
for name, stopping in (("Suzuki", suzuki), ("table", table)):
    results[name] = foc.deposit(
        beam,
        equilibrium,
        plasma,
        stopping,
        step=0.03,
        births_per_marker=0,
        rho_edges=rho_edges,
    )
    print(
        f"{name:6s}: shine-through {100 * results[name].shine_through_fraction:.3f} %"
    )
shine = [results[name].shine_through_fraction for name in results]
power = [results[name].power for name in results]
print(f"shine-through differs by {100 * (shine[1] / shine[0] - 1):.3f} %")
print(
    f"power per shell by at most {100 * np.max(np.abs(power[1] / power[0] - 1)):.3f} %"
)
assert abs(shine[1] / shine[0] - 1) < 5e-3
np.testing.assert_allclose(power[1], power[0], rtol=5e-3)

surfaces = equilibrium.flux_surfaces(rho_edges)
power_density = xr.concat(
    [results[name].to_dataset(surfaces).power_density for name in results],
    dim=xr.DataArray(list(results), dims="model", name="model"),
)
power_density.plasma.plot.profiles(x="rho", over="model", title="")

The two depositions agree to the interpolation error of the table: the two power-density profiles lie on each other.

## ADAS adf21 files

ADAS tabulates the effective beam-stopping coefficient $S = \sigma v_\mathrm{beam}$ [cm$^3$/s] of one receiver per adf21 file, as a table in beam energy and density at a reference temperature $T_\mathrm{ref}$, and a temperature dependence at a reference energy and density. Between them it is the separable product

$$S(E, n, T) = \frac{S_{En}(E, n)\, S_T(T)}{S_\mathrm{ref}}.$$

ADAS data are licensed, so none comes with freeofcharge. To show the format, the cell below writes two synthetic files with `write_adf21`, from Suzuki's pure-plasma tables of deuterium and carbon (the coefficients rounded to four digits, as in the files), and reads them back. With real files, pass their paths to `TabulatedStopping.from_adf21`.

In [ ]:
def synthetic_adf21(index, charge, element, t_ref=5e3, e_ref=1e5, n_ref=5e19):
    """An adf21 file of the table's species ``index`` (cm^3/s, cm^-3)."""
    e_grid = log_grid(10e3, 5e6, 8, (1e5 / 1.1, 1.1e5))
    n_grid = log_grid(1e18, 3e20, 4)
    t_grid = log_grid(100.0, 3e4, 8)
    speed = np.sqrt(2.0 * e_grid * 1.602176634e-19 / 1.66053906660e-27)
    s_en = table.species_sigma(e_grid[:, None], n_grid[None, :], t_ref)[..., index]
    s_t = table.species_sigma(e_ref, n_ref, t_grid)[..., index]
    v_ref = np.sqrt(2.0 * e_ref * 1.602176634e-19 / 1.66053906660e-27)
    s_ref = table.species_sigma(e_ref, n_ref, t_ref)[index] * v_ref * 1e6
    return Adf21(
        charge=charge,
        element=element,
        energies=e_grid,
        densities=n_grid * 1e-6,
        coefficients=s_en * speed[:, None] * 1e6,  # m^3/s -> cm^3/s
        temperatures=t_grid,
        temperature_coefficients=s_t * v_ref * 1e6,
        reference=s_ref,
        reference_energy=e_ref,
        reference_density=n_ref * 1e-6,
        reference_temperature=t_ref,
    )


paths = {"D": folder / "synthetic_d1.dat", "C": folder / "synthetic_c6.dat"}
write_adf21(paths["D"], synthetic_adf21(0, 1, "H"), comment="synthetic, from Suzuki")
write_adf21(paths["C"], synthetic_adf21(3, 6, "C"), comment="synthetic, from Suzuki")
print("\n".join(paths["C"].read_text().splitlines()[:8]))

adas = TabulatedStopping.from_adf21(paths)  # density_rule "equivalent"
data = read_adf21(paths["C"])
print(
    f"carbon: {len(data.energies)} energies, {len(data.densities)} densities, "
    f"{len(data.temperatures)} temperatures, S_ref = {data.reference:.3e} cm^3/s"
)

The file's header holds the receiver's charge and $S_\mathrm{ref}$, then come the energy and density grids, the table, and the temperature dependence. The adf21 table reproduces the file's values at its nodes; between them it is the separable product. The rounding of the file and the separable product cost some accuracy against the full Suzuki table. The density rule matters for the impurity: ADAS reads carbon's table at $\sum_j Z_j^2 n_j / 6$, the equivalent electron density of a pure carbon plasma with the same ion collisions, $Z_\mathrm{eff} n_e / 6$. The figure compares the synthetic adf21 tables, combined with either rule, with Suzuki's fit in a deuterium plasma with 2 % of the electrons from carbon, at $n_e = 5 \times 10^{19}\,\mathrm{m^{-3}}$ and $T$ = 3 keV:

In [ ]:
adas_electron = TabulatedStopping(adas.table, density_rule="electron")
e_line = np.geomspace(20e3, 2e6, 80)
ne = 5e19
args = (e_line, ne, 3e3, *ion_arguments(ne, ions))
sigma = {
    "Suzuki": np.asarray(suzuki(*args)),
    "adf21, electron rule": np.asarray(adas_electron(*args)),
    "adf21, equivalent rule (ADAS)": np.asarray(adas(*args)),
}
for name in list(sigma)[1:]:
    rel = np.abs(sigma[name] / sigma["Suzuki"] - 1)
    print(f"{name}: within {100 * rel.max():.2f} % of Suzuki")
assert np.max(np.abs(sigma["adf21, electron rule"] / sigma["Suzuki"] - 1)) < 0.02

difference = xr.DataArray(
    [100 * (sigma[name] / sigma["Suzuki"] - 1) for name in list(sigma)[1:]],
    dims=("rule", "energy"),
    coords={
        "rule": ("rule", list(sigma)[1:]),
        "energy": (
            "energy",
            e_line / 1e3,
            {"long_name": "beam energy", "units": "keV/amu"},
        ),
    },
    name="adf21 / Suzuki - 1",
    attrs={"units": "%"},
)
difference.plasma.plot.profiles(x="energy", over="rule", logx=True, title="")

With the electron rule, the rule Suzuki's fit was written in, the synthetic files stay within 0.7 % of the fit. That is mostly the cost of the separable product: the temperature dependence is taken at $E_\mathrm{ref}$ = 100 keV/amu and applied at every energy, while the fit's temperature dependence differs between its low- and high-energy parts, hence the step at 100 keV/amu ($T$ = 3 keV here, $T_\mathrm{ref}$ = 5 keV). The equivalent rule moves the result by up to another half per cent, through the density at which it reads carbon's table. With real ADAS files, the equivalent rule is the one they are made for.

Outside its grid a table raises a ValueError by default; `outside="clamp"` holds the edge value and `outside="extrapolate"` continues the last cell, for all axes or per axis (`outside={"density": "clamp"}`).